# Types of Runnable

Runnables can be classified into two types:
1) Task Specific Runnables
2) Primitive Runnables


* `Task Specific Runnables`: Those classes which are created to perform some task but for our convenience, we convert these methods to runnable, such are called Task Specific Runnables.
`For example`: ChatOpenAI, GoogleGenerativeAI, PromptTemplate etc. All these are classes but converted to Runnable and inserted invoke() menthods in all.

* Primitive Runnables: These are core runnables which is ceated to help us ease some task.
`For example`: RunnableSequence, RunnableParallel, RunnablePassThrough etc.

# 1. Runnable Sequence

In [7]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain.schema.runnable import RunnableSequence, RunnableParallel

In [8]:
load_dotenv()

True

In [9]:
model_gemini = GoogleGenerativeAI(
    model = 'gemini-2.5-flash'
)

temp = PromptTemplate(
    template = "Generate 2 paragraphs on {topic}",
    input_variables = ['topic']
)

parser = StrOutputParser()

chain = RunnableSequence(temp, model_gemini, parser)

response = chain.invoke({'topic': 'India'})

print(response)

India, a land steeped in millennia of history, is a vibrant tapestry woven from countless cultures, languages, and spiritual traditions. From the snow-capped peaks of the Himalayas to the sun-drenched beaches of the south, its vast landscape cradles an unparalleled diversity, being the birthplace of Hinduism and Buddhism, and home to thriving communities of Islam, Christianity, Sikhism, and Jainism. This rich mosaic manifests in an explosion of sensory experiences: the aromatic spices of its diverse cuisines, the kaleidoscope of colors in its festivals, the intricate beauty of its ancient temples and forts, and the rhythmic melodies of its classical music and dance, all speaking to a civilization that has continuously evolved while preserving its profound heritage.

Beyond its ancient soul, India is a dynamic and rapidly modernizing nation, asserting its place on the global stage as the world's most populous democracy. It boasts a burgeoning economy driven by a youthful and ambitious p

In [10]:
temp2 = PromptTemplate(
    template = '''GEnerate a 5 lines summary on below given paragraph
    {para}''',
    input_variables = ['para']
)

chain = RunnableSequence(temp, model_gemini, parser, temp2, model_gemini, parser)

print(chain.invoke({'topic': 'India'}))

India is an ancient, immensely diverse subcontinent, defined by varied landscapes, countless languages, and myriad faiths. Its millennia-old civilization has profoundly shaped global culture through rich spiritual and artistic traditions.

Today, it stands as the world's largest democracy, navigating rapid development and economic growth. A youthful population, entrepreneurial spirit, and IT prowess drive its emergence as a global powerhouse. This unique blend of ancient wisdom and modern ambition defines India's resilient journey on the global stage.


# 2) Runnable Parallel

Suppose we have two different LLMs. One can generate good LinkedIn post and another LLM can generate good tweet. To do so, we have `RunnableParallel` which runnes parallely. 

Both models gets same input and does different job assigned to them.

In [11]:
temp_tweet = PromptTemplate(
    template = "Generate a tweet about {topic}",
    input_variables = ['topic']
)

temp_linkedin = PromptTemplate(
    template = "Generate a LinkedIn post about {topic}",
    input_variable = ['topic']
)

parallel_chain = RunnableParallel(
    {'tweet': RunnableSequence(temp_tweet, model_gemini, parser),
    'linkedin': RunnableSequence(temp_linkedin, model_gemini, parser)}
)

response = parallel_chain.invoke({'topic': 'India'})

print(response)

{'tweet': "Here are a few options for a tweet about India, depending on the angle you want to take:\n\n**Option 1 (Focus on Vibrancy/Culture):**\n> Immerse yourself in the vibrant tapestry of India! From ancient traditions to bustling modern cities, every corner tells a unique story. ✨ #IncredibleIndia #Culture #Diversity\n\n**Option 2 (Focus on Innovation/Future):**\n> India: A global hub for tech innovation and a thriving startup ecosystem! The future is being built here, fueled by creativity and ambition. 🚀 #DigitalIndia #Innovation #StartupNation\n\n**Option 3 (Focus on Travel/Experience):**\n> Dream of an unforgettable journey? India offers a kaleidoscope of experiences – from majestic mountains to serene beaches, bustling markets to spiritual retreats. 🇮🇳 #TravelIndia #Wanderlust #Explore\n\n**Option 4 (Focus on Food/Flavors):**\n> Exploring the incredible flavors of India! From rich curries to street food delights, every bite is an adventure. What's your favorite Indian dish? 🌶️

In [12]:
for key, value in response.items():
    print(f'{key}: {value}')
    print()

tweet: Here are a few options for a tweet about India, depending on the angle you want to take:

**Option 1 (Focus on Vibrancy/Culture):**
> Immerse yourself in the vibrant tapestry of India! From ancient traditions to bustling modern cities, every corner tells a unique story. ✨ #IncredibleIndia #Culture #Diversity

**Option 2 (Focus on Innovation/Future):**
> India: A global hub for tech innovation and a thriving startup ecosystem! The future is being built here, fueled by creativity and ambition. 🚀 #DigitalIndia #Innovation #StartupNation

**Option 3 (Focus on Travel/Experience):**
> Dream of an unforgettable journey? India offers a kaleidoscope of experiences – from majestic mountains to serene beaches, bustling markets to spiritual retreats. 🇮🇳 #TravelIndia #Wanderlust #Explore

**Option 4 (Focus on Food/Flavors):**
> Exploring the incredible flavors of India! From rich curries to street food delights, every bite is an adventure. What's your favorite Indian dish? 🌶️ #IndianCuisine 

# 3) RunnablePassThrough

Runnable PassThrough returns input as it is as output. It doesn't do any processing on input, it output the input as it is. <br>
`For example`: <br>
Input = 'Generate 5 lines on India' <br>
Output = 'Generate 5 lines on India'

In [13]:
from langchain.schema.runnable import RunnableSequence, RunnableParallel, RunnablePassthrough

In [14]:
runnable_pass = RunnablePassthrough()

runnable_pass.invoke(['Nikunj'])   # See, it is returning the input as it is

['Nikunj']

In [15]:
temp1 = PromptTemplate(
    template = "Generate a joke on {topic}",
    input_variables = ['topic']
)

temp2 = PromptTemplate(
    template = "Explain the given joke: {joke}",
    input_variables = ['joke']
)

In [22]:
joke_chain = RunnableSequence(temp1, model_gemini, parser)

parallel_chain = RunnableParallel(
    {'joke': RunnablePassthrough(),
    'explanation': RunnableSequence(temp2, model_gemini, parser)}
)

combined_chain = RunnableSequence(joke_chain, parallel_chain)

response = combined_chain.invoke({'topic': 'cricket'})



{'joke': 'Why did the cricket ball break up with the bat?\n\nBecause it felt like their relationship was going on for an eternity, and it just wanted to get bowled over by someone new!', 'explanation': 'This joke plays on two key elements: **anthropomorphism** (giving human qualities to inanimate objects) and **puns** related to cricket.\n\nHere\'s the breakdown:\n\n1.  **"Why did the cricket ball break up with the bat?"**\n    *   This sets up the anthropomorphism. A ball and bat can\'t have a relationship, much less break up. This is where the humor begins.\n\n2.  **"Because it felt like their relationship was going on for an eternity..."**\n    *   In real cricket, the contact between the ball and the bat is incredibly brief – a split second.\n    *   The humor here comes from the absurd contrast: the ball, experiencing countless brief, repetitive interactions with the bat (being hit over and over), perceives this endless cycle as an "eternity." It\'s tired of the same old routine.\

In [34]:
print(f"joke: {response['joke']},\n\n\n explanation: {response['explanation']}")

joke: Why did the cricket ball break up with the bat?

Because it felt like their relationship was going on for an eternity, and it just wanted to get bowled over by someone new!,


 explanation: This joke plays on two key elements: **anthropomorphism** (giving human qualities to inanimate objects) and **puns** related to cricket.

Here's the breakdown:

1.  **"Why did the cricket ball break up with the bat?"**
    *   This sets up the anthropomorphism. A ball and bat can't have a relationship, much less break up. This is where the humor begins.

2.  **"Because it felt like their relationship was going on for an eternity..."**
    *   In real cricket, the contact between the ball and the bat is incredibly brief – a split second.
    *   The humor here comes from the absurd contrast: the ball, experiencing countless brief, repetitive interactions with the bat (being hit over and over), perceives this endless cycle as an "eternity." It's tired of the same old routine.

3.  **"...and it j

In [41]:
print(f"""joke: {response['joke']},


explanation: {response['explanation']}""")

joke: Why did the cricket ball break up with the bat?

Because it felt like their relationship was going on for an eternity, and it just wanted to get bowled over by someone new!,


explanation: This joke plays on two key elements: **anthropomorphism** (giving human qualities to inanimate objects) and **puns** related to cricket.

Here's the breakdown:

1.  **"Why did the cricket ball break up with the bat?"**
    *   This sets up the anthropomorphism. A ball and bat can't have a relationship, much less break up. This is where the humor begins.

2.  **"Because it felt like their relationship was going on for an eternity..."**
    *   In real cricket, the contact between the ball and the bat is incredibly brief – a split second.
    *   The humor here comes from the absurd contrast: the ball, experiencing countless brief, repetitive interactions with the bat (being hit over and over), perceives this endless cycle as an "eternity." It's tired of the same old routine.

3.  **"...and it ju

# 4) RunnableLambda: 

It convert any Python function to a runnable <br>
`For example:`

Supose we have feedbacks of customers stored in our database and we are passing it to an LLM to get the sentiment.
But the problem here is, the feedbacks are not well structured, there are HTML tags, emojies, puntuations marks etc.

So we created a function which does all the preprocessing steps and then fo to LLM.

With `RunnableLambda`, we can convert this function to a runnable and then intigrate it with a chain.

`Task:`

We will create a chain which will give us a joke on some topic and also return no_of_words present in the joke.

To do so, we will create RunnableParallel. <br>
One model will genertae joke and anothere will be RunnableLambda which will return no_of_words present in it.

In [45]:
from langchain.schema.runnable import RunnableLambda, RunnableParallel, RunnableSequence

In [46]:
def word_count(string):
    l = string.split()
    return len(l)

In [52]:
word_count("My name is Nikunj")

4

In [69]:
# converting word_count to a runnable

runnable_word_count = RunnableLambda(word_count)
runnable_word_count.invoke("My name is Nikunj")

4

In [73]:
temp1 = PromptTemplate(
    template = "Generate a joke on {topic}",
    input_variables = ['topic']
)

joke_chain = RunnableSequence(temp1, model_gemini, parser)

parallel_runnable = RunnableParallel(
    {'joke': RunnablePassthrough(),
    'no_of_words': RunnableLambda(word_count)}
)

combined_runnable = RunnableSequence(joke_chain, parallel_runnable)

response = combined_runnable.invoke({'topic': 'Cricket'})

In [74]:
print(f"""joke: {response['joke']}

No of words: {response['no_of_words']}""")

joke: Why did the new spectator get confused during his first cricket match?

He watched for hours, then turned to his friend and asked, "So, when do they finally decide a winner?"

His friend replied, "Oh, this is just the *first day*. We still have four more days, and then maybe, just maybe, they'll break for lunch!"

No of words: 56


# 5) RunnableBrance:

`RunnableBranch` is used to create conditional chains.

`For example:` <br>
Supose a user send us an email and we want such a model that if an email is about any complain, connect to cuatomer care.<br>
If an email is regarding refund, connect to database <br>
and if an email is general query, connect it to the chatbot.

`Task:`

Create a model which generates a report on some topic. If the topic is more than 500 words, get it summarize else leave it as it is.

In [85]:
from langchain.schema.runnable import RunnableSequence, RunnableParallel, RunnablePassthrough, RunnableLambda, RunnableBranch

In [110]:
temp1 = PromptTemplate(
    template = 'Generate a detail report on {topic}',
    input_variables = ['topic']
)

temp2 = PromptTemplate(
    template = "Generate a 5 points summary of the given report: {report}",
    input_variables = ['report']
)

def word_count(text):

    return len(text.split())


report_chain = RunnableSequence(temp1, model_gemini, parser)

branch_chain = RunnableBranch(
    (RunnableLambda(word_count) > 250, temp2 | model_gemini | parser),
    (RunnablePassthrough())
)

# This will give error bcz RunnableLambda(word_count) will convert word_count() to a runnable object and then will compare using > 250.
# So, this will give error.

TypeError: '>' not supported between instances of 'RunnableLambda' and 'int'

In [111]:
branch_chain = RunnableBranch(
    (RunnableLambda(lambda x: len(x.split()) > 100), temp2 | model_gemini | parser),
    RunnablePassthrough()
)

combined_chain = RunnableSequence(report_chain, branch_chain, parser)

response = combined_chain.invoke(
    {'topic': 'India'}
)

In [112]:
print(response)

Here is a 5-point summary of the detailed report on India:

1.  **Vast Diversity and Scale:** India is the world's most populous and largest democracy, characterized by immense geographic, cultural, linguistic, and religious diversity, often described as a subcontinent.
2.  **Rapidly Growing Economy:** It possesses a rapidly expanding mixed economy, predominantly driven by its robust services sector (especially IT), a large domestic market, and a significant young workforce, positioning it as a major global economic power.
3.  **Democratic Governance & Complex Society:** As a sovereign, democratic republic operating under a parliamentary system, India embodies "unity in diversity" while navigating a complex social fabric influenced by historical divisions and evolving societal norms.
4.  **Significant Socio-Economic & Environmental Challenges:** Despite its growth, India faces substantial challenges including widespread poverty, income inequality, high unemployment, infrastructure defi

In [109]:
len(response.split())

345